In [1]:
from helper_functions import sensitivity_quad_unif
from helper_functions import construct_E_C
from helper_functions import sens_recovery
from helper_functions import LSE
from helper_functions import construct_R
from helper_functions import decomp_non_neg_garrote
from helper_functions import decomp_orthog
from sklearn.cluster import SpectralCoclustering
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [2]:
N = 30
rng = np.random.default_rng(10)
inputs = rng.uniform(0.0,1.0,size = (N,5))
model_matrix = np.concatenate((inputs,inputs**2),axis = 1)
print(model_matrix)

a=0

[[9.56001710e-01 2.07681810e-01 8.28444885e-01 1.49282123e-01
  5.12804616e-01 9.13939269e-01 4.31317342e-02 6.86320928e-01
  2.22851523e-02 2.62968575e-01]
 [1.35919604e-01 6.89036480e-01 8.41747724e-01 4.25508997e-01
  9.56926003e-01 1.84741388e-02 4.74771270e-01 7.08539231e-01
  1.81057907e-01 9.15707376e-01]
 [8.25332906e-01 3.38215312e-01 5.75760548e-01 7.53301865e-01
  8.27103937e-01 6.81174406e-01 1.14389598e-01 3.31500209e-01
  5.67463699e-01 6.84100923e-01]
 [9.33438471e-01 1.44994695e-01 7.45580211e-01 1.39351394e-01
  9.06528756e-01 8.71307379e-01 2.10234616e-02 5.55889851e-01
  1.94188110e-02 8.21794386e-01]
 [2.26114434e-01 8.53239750e-01 3.06317866e-01 9.69830368e-01
  5.17834214e-01 5.11277371e-02 7.28018071e-01 9.38306347e-02
  9.40570943e-01 2.68152273e-01]
 [3.22474559e-01 2.82433517e-01 6.05864996e-01 3.33764457e-01
  6.78648774e-01 1.03989841e-01 7.97686916e-02 3.67072393e-01
  1.11398713e-01 4.60564158e-01]
 [1.54425070e-01 2.49775519e-01 8.69894246e-01 6.00367820e

In [3]:
def function_responses(model_input):

    resp_1 = np.dot(np.array([-1,1,a,0,0,-1,1,a/2,0,0]),model_input) + rng.normal(0,0.1)
    resp_2 = np.dot(np.array([1,-1,a,0,0,1,-1,a/2,0,0]),model_input) + rng.normal(0,0.1)
    resp_3 = np.dot(np.array([0,0,a/2,1,-1,0,0,a/4,1,-1]),model_input) + rng.normal(0,0.1)
    resp_4 = np.dot(np.array([0,0,a/2,-1,1,0,0,a/4,-1,1]),model_input) + rng.normal(0,0.1)

    return[resp_1,resp_2,resp_3,resp_4]

In [4]:
responses = np.zeros((4,N))
for n in range(N):
    four_resp = function_responses(model_matrix[n])
    responses[0,n] = four_resp[0]
    responses[1,n] = four_resp[1]
    responses[2,n] = four_resp[2]
    responses[3,n] = four_resp[3]
print(responses)

[[-1.64743993  1.15802288 -1.10936865 -1.67272587  1.04966301 -0.14429403
   0.04253751 -0.16315375 -0.15583696  0.19793214  0.89624557 -0.98241086
   1.23817025  1.04658509  0.40294981 -0.44969913  0.14238758 -0.11312769
   0.14270699  0.81337448  1.04770874 -0.95368248 -0.46109303  0.56947546
   0.87004377 -0.5692122   0.02935444 -1.30147256 -0.47825924 -1.07099449]
 [ 1.60735712 -1.26348935  0.98319661  1.7375302  -1.31484461 -0.03991859
  -0.42212597 -0.27719416  0.17573828 -0.25229958 -1.06880474  0.79628535
  -1.21983863 -0.95972995 -0.3054984   0.33413162 -0.23177942  0.16854207
  -0.1335823  -0.76338888 -1.17419319  0.97301428  0.63258018 -0.54269228
  -0.90000076  0.3298451   0.01273949  1.05677761  0.48736649  1.04573512]
 [-0.47400819 -1.31275873 -0.21368254 -1.59820441  1.16481702 -0.504535
   0.45391519 -0.59079676 -0.576725    0.76087836 -0.15361684  0.31868131
  -0.10599694 -0.05924079 -0.4479375   0.39049623 -1.0501356   0.46896424
   0.72867259  0.02141114  0.48506641 

In [5]:
LSE_1 = LSE(model_matrix,responses[0].T)
print(LSE_1)
LSE_2 = LSE(model_matrix,responses[1].T)
print(LSE_2)
LSE_3 = LSE(model_matrix,responses[2].T)
print(LSE_3)
LSE_4 = LSE(model_matrix,responses[3].T)
print(LSE_4)

[-0.43235602  0.99254384 -0.24771875  0.03583833 -0.19784327 -1.59118892
  0.93851177  0.19977804 -0.24342393  0.32062565]
[ 1.92152136 -1.13137337 -0.28026965 -0.54598262 -0.2634899   0.24214139
 -0.83789911  0.26151815  0.45306018  0.23323634]
[ 0.6668532  -0.27588503 -0.01261189  0.78474335 -1.00823377 -0.58176563
  0.14164689 -0.09616669  1.22327871 -0.91471627]
[-0.35324401  0.12766489  0.15749921 -0.95385876  0.9118459   0.40885437
 -0.04539289 -0.18689214 -1.01274089  0.9769488 ]


In [6]:
sens_lse_1 = [LSE_1[0],LSE_1[5],LSE_1[1],LSE_1[6],LSE_1[2],LSE_1[7],LSE_1[3],LSE_1[8],LSE_1[4],LSE_1[9]]
sens_lse_2 = [LSE_2[0],LSE_2[5],LSE_2[1],LSE_2[6],LSE_2[2],LSE_2[7],LSE_2[3],LSE_2[8],LSE_2[4],LSE_2[9]]
sens_lse_3 = [LSE_3[0],LSE_3[5],LSE_3[1],LSE_3[6],LSE_3[2],LSE_3[7],LSE_3[3],LSE_3[8],LSE_3[4],LSE_3[9]]
sens_lse_4 = [LSE_4[0],LSE_4[5],LSE_4[1],LSE_4[6],LSE_4[2],LSE_4[7],LSE_4[3],LSE_4[8],LSE_4[4],LSE_4[9]]

list_of_sens_score = [sensitivity_quad_unif(sens_lse_1),sensitivity_quad_unif(sens_lse_2),sensitivity_quad_unif(sens_lse_3),sensitivity_quad_unif(sens_lse_4)]
print(list_of_sens_score)

[[0.524731849203479, 0.4661692334619911, 0.0006103347059722708, 0.005789691734562916, 0.002698890893994564], [0.5421625862581758, 0.45416095740356205, 0.0005682820202349748, 0.0025826115120104473, 0.00052556280601668], [0.003750240049435667, 0.002435808166478473, 0.0015665347507443253, 0.5199310632165646, 0.47231635381677695], [0.0018753641882827696, 0.0009097202601215226, 0.0004205445867975478, 0.5184670299689201, 0.47832734099587787]]


In [7]:
clustering = SpectralCoclustering(n_clusters=2, random_state=0).fit(list_of_sens_score)
clustering.biclusters_

C:\Users\wsfishe\Anaconda3\lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


(array([[ True,  True, False, False],
        [False, False,  True,  True]]),
 array([[ True,  True, False, False, False],
        [False, False,  True,  True,  True]]))

In [8]:
#START THE NON-NEGATIVE GARROTE ANALYSIS
#NEED TO CONSTRUCT R
R_1 = construct_R(model_matrix,responses[0].T)[0]
print(R_1)

R_2 = construct_R(model_matrix,responses[1].T)[0]
print(R_2)

R_3 = construct_R(model_matrix,responses[2].T)[0]
print(R_3)

R_4 = construct_R(model_matrix,responses[3].T)[0]
print(R_4)

[[-4.13333092e-01  2.06133301e-01 -2.05221329e-01  5.35002160e-03
  -1.01454940e-01 -1.45425003e+00  4.04796404e-02  1.37111849e-01
  -5.42473924e-03  8.43144691e-02]
 [-5.87656586e-02  6.83898912e-01 -2.08516692e-01  1.52495308e-02
  -1.89321366e-01 -2.93958448e-02  4.45578427e-01  1.41550578e-01
  -4.40738264e-02  2.93599269e-01]
 [-3.56837648e-01  3.35693524e-01 -1.42626682e-01  2.69970789e-02
  -1.63636944e-01 -1.08387717e+00  1.07355984e-01  6.62264615e-02
  -1.38134241e-01  2.19340300e-01]
 [-4.03577739e-01  1.43913591e-01 -1.84694196e-01  4.99412088e-03
  -1.79350610e-01 -1.38641464e+00  1.97307662e-02  1.11054584e-01
  -4.72700319e-03  2.63488356e-01]
 [-9.77619360e-02  8.46877856e-01 -7.58806779e-02  3.47570983e-02
  -1.02450012e-01 -8.13538886e-02  6.83253531e-01  1.87453002e-02
  -2.28957471e-01  8.59764956e-02]
 [-1.39423816e-01  2.80327647e-01 -1.50084118e-01  1.19615599e-02
  -1.34266090e-01 -1.65467482e-01  7.48638562e-02  7.33330028e-02
  -2.71171119e-02  1.47668681e-01

In [9]:
#Constuct clusters. J = 4 K = 2. There should only be 3 in this case. Z_1 is the correct clustering
Z_1 = np.array([[1,0],
      [1,0],
      [0,1],
      [0,1]])
Z_2 = np.array([[1,0],
      [0,1],
      [1,0],
      [0,1]])
Z_3 = np.array([[1,0],
      [0,1],
      [0,1],
      [1,0]])

#Construct Strong Heredity List. If a quadratic term is included, then the corresponding linear term should be included.
D_str_her = [[[],[],[],[],[],[0],[1],[2],[3],[4]],
    [[],[],[],[],[],[0],[1],[2],[3],[4]],
    [[],[],[],[],[],[0],[1],[2],[3],[4]],
    [[],[],[],[],[],[0],[1],[2],[3],[4]],]

gamma_test = 1
lam_test = 0.0
kappa_test = 2
print(Z_1[0,0])

1


In [10]:
garrote = decomp_non_neg_garrote(responses,[R_1,R_2,R_3,R_4],Z_1,D_str_her,gamma_test,lam_test,kappa_test,t = 50)
print(garrote)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-05-09
Set parameter TimeLimit to value 50
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i9-11900H @ 2.50GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 20 rows, 40 columns and 40 nonzeros
Model fingerprint: 0x51a3cdd3
Model has 260 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 2e+01]
  QObjective range [4e-03, 7e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]

Continuous model is non-convex -- solving as a MIP

Found heuristic solution: objective 72.3374787
Presolve time: 0.00s
Presolved: 461 rows, 302 columns, 1060 nonzeros
Presolved model has 40 quadratic constraint(s)
Presolved model has 220 bilinear constraint(s)
         in product terms.
 

In [11]:
orthog = decomp_orthog(responses,[R_1,R_2,R_3,R_4],D_str_her,2,t=50)
print(orthog)

Set parameter TimeLimit to value 50
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i9-11900H @ 2.50GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 26 rows, 60 columns and 56 nonzeros
Model fingerprint: 0x981522c0
Model has 220 quadratic objective terms
Model has 18 quadratic constraints
Variable types: 46 continuous, 14 integer (14 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [2e-02, 2e+01]
  QObjective range [4e-03, 7e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]
Presolve added 20 rows and 0 columns
Presolve removed 0 rows and 4 columns
Presolve time: 0.00s
Presolved: 292 rows, 117 columns, 606 nonzeros
Presolved model has 6 SOS constraint(s)
Presolve

In [ ]:
def L_2_norm_metric(inflation_factors,est_LSEs,true_LSEs):
    #inflation_factors: a list/numpy array of numpy vectors of inflation factors
    #est_LSEs: a list/numpy array of numpy vectors of estimated LSEs
    #true_LSEs: a list/numpy array of numpy vectors of true_LSEs
    print([inflation_factors[i]*est_LSEs[i] - true_LSEs[i] for i in range(len(inflation_factors))])
    L_2_norms = [ np.sqrt(np.dot(inflation_factors[i]*est_LSEs[i] - true_LSEs[i],inflation_factors[i]*est_LSEs[i] - true_LSEs[i])) for i in range(len(inflation_factors))]

    return L_2_norms

In [ ]:
#a=0 Run this cell only when a=0!
inflation_bipartite_0 = np.array([
    [1,1,0,0,0,1,1,0,0,0],
    [1,1,0,0,0,1,1,0,0,0],
    [0,0,1,1,1,0,0,1,1,1],
    [0,0,1,1,1,0,0,1,1,1]
])

print('bipartite_0:' + str(L_2_norm_metric(inflation_bipartite_0,np.array([LSE_1,LSE_2,LSE_3,LSE_4]),np.array([[-1,1,a,0,0,-1,1,a/2,0,0],[1,-1,a,0,0,1,-1,a/2,0,0],[0,0,a/2,1,-1,0,0,a/4,1,-1],[0,0,a/2,-1,1,0,0,a/4,-1,1]]))))
print('garrote_0:' + str(L_2_norm_metric(garrote[0],np.array([LSE_1,LSE_2,LSE_3,LSE_4]),np.array([[-1,1,a,0,0,-1,1,a/2,0,0],[1,-1,a,0,0,1,-1,a/2,0,0],[0,0,a/2,1,-1,0,0,a/4,1,-1],[0,0,a/2,-1,1,0,0,a/4,-1,1]]))))

In [ ]:
#a=1 Run this cell only when a=1!
inflation_bipartite_1 = np.array([
    [1,1,1,0,0,1,1,1,0,0],
    [1,1,1,0,0,1,1,1,0,0],
    [0,0,0,1,1,0,0,0,1,1],
    [0,0,0,1,1,0,0,0,1,1]
])

print('bipartite_1:' + str(L_2_norm_metric(inflation_bipartite_1,np.array([LSE_1,LSE_2,LSE_3,LSE_4]),np.array([[-1,1,a,0,0,-1,1,a/2,0,0],[1,-1,a,0,0,1,-1,a/2,0,0],[0,0,a/2,1,-1,0,0,a/4,1,-1],[0,0,a/2,-1,1,0,0,a/4,-1,1]]))))
print('garrote_1:' + str(L_2_norm_metric(garrote[0],np.array([LSE_1,LSE_2,LSE_3,LSE_4]),np.array([[-1,1,a,0,0,-1,1,a/2,0,0],[1,-1,a,0,0,1,-1,a/2,0,0],[0,0,a/2,1,-1,0,0,a/4,1,-1],[0,0,a/2,-1,1,0,0,a/4,-1,1]]))))